In [71]:
# import pandas as pd
# import numpy as np

# # 1. Load Dataset Final
# df_final = pd.read_csv('../../datas/dataset_final.csv')
# print(f"Data Loaded: {df_final.shape}")

# print(df_final.info())

# # 2. Siapkan "Database Profil User"
# # Kita butuh daftar unik setiap user beserta atribut fisiknya untuk dicocokkan
# # Kita pakai kolom dengan akhiran _x karena itu biasanya data profil awal
# cols_profile = [
#     'User_ID', 'Age_x', 'Gender_x', 'Height_cm_x', 'Initial_Weight_kg_x', 
#     'Goal_x', 'Workout_Frequency_x', 'level_x', 'Environment'
# ]

# # Ambil 1 baris per user saja (Drop duplikat karena User_ID berulang di dataset_final)
# df_users_db = df_final[cols_profile].drop_duplicates(subset=['User_ID']).copy()

# print(f"Jumlah User Unik di Database: {len(df_users_db)}")

# # --- FUNGSI 1: CARI KEMBARAN (SIMILARITY) ---
# def find_similar_user(new_user_data):
#     """
#     Mencari User_ID dari database yang paling mirip dengan user baru.
#     """
#     candidates = df_users_db.copy()
    
#     # FILTER 1: Cari yang Goal-nya SAMA (Wajib)
#     # Kalau goal beda (misal user mau Muscle Gain tapi dikasih Weight Loss), jadwalnya pasti salah.
#     candidates = candidates[candidates['Goal_x'] == new_user_data['Goal']]
    
#     # FILTER 2: Cari yang Frekuensinya SAMA (Wajib)
#     # Biar jadwalnya pas (misal user minta 3 hari, dikasih jadwal 3 hari)
#     candidates = candidates[candidates['Workout_Frequency_x'] == new_user_data['Workout_Frequency'] & candidates['Workout_Frequency_x'] == new_user_data['Workout_Frequency']] 
    
#     # Jika tidak ada yang cocok persis, kita longgarkan filter (ambil Goal aja)
#     if candidates.empty:
#         print("Note: Tidak menemukan frekuensi yang sama persis, mencari berdasarkan Goal saja...")
#         candidates = df_users_db[df_users_db['Goal_x'] == new_user_data['Goal']].copy()
    
#     # FILTER 3: Hitung Skor Kemiripan (Similarity Score)
#     # Semakin kecil skor, semakin mirip.
#     # Kita bandingkan Umur, Tinggi, dan Berat.
    
#     # Kita ubah Gender jadi angka dulu (0/1) buat hitungan
#     gender_input = 1 if new_user_data['Gender'] == 'Male' else 0
#     candidates['Gender_Num'] = candidates['Gender_x'].apply(lambda x: 1 if x == 'Male' else 0)
    
#     candidates['Similarity_Score'] = (
#         abs(candidates['Age_x'] - new_user_data['Age']) * 1 +            # Bobot Umur: 1
#         abs(candidates['Height_cm_x'] - new_user_data['Height']) * 2 +   # Bobot Tinggi: 2
#         abs(candidates['Initial_Weight_kg_x'] - new_user_data['Weight']) * 2 + # Bobot Berat: 2
#         abs(candidates['Gender_Num'] - gender_input) * 50                # Bobot Gender: 50 (Harus sama!)
#     )
    
#     # Urutkan dari skor terkecil (paling mirip)
#     best_match = candidates.sort_values('Similarity_Score').iloc[0]
    
#     return best_match['User_ID'], best_match['Similarity_Score']

# # --- FUNGSI 2: AMBIL JADWAL ---
# def get_workout_plan(user_id_target):
#     """
#     Mengambil semua jadwal latihan milik User ID tertentu.
#     """
#     # Ambil baris data milik user tersebut
#     schedule = df_final[df_final['User_ID'] == user_id_target].copy()
    
#     # Pilih kolom yang mau ditampilkan sebagai jadwal
#     cols_display = [
#         'Day', 'Muscle Group', 'Exercise Name', 
#         'Equipment', 'Sets', 'Reps', 'Instructions'
#     ]
    
#     # Rapikan: Urutkan berdasarkan Hari, lalu Buang duplikat latihan (kalau ada)
#     # Kita asumsikan kolom 'Day' bisa diurutkan string-nya (Day 1, Day 2...)
#     schedule_clean = schedule[cols_display].drop_duplicates()
    
#     return schedule_clean

# # --- CONTOH PEMAKAIAN ---

# # 1. Data User Baru (Input dari Frontend/Aplikasi)
# new_user = {
#     'Age': 25,
#     'Gender': 'Male',
#     'Height': 175,
#     'Weight': 70,
#     'Goal': 'Muscle Gain',        # Harus sama persis tulisannya dengan di CSV
#     'Workout_Frequency': 4,       # Mau latihan 4 hari
#     'Level': 'Beginner',
#     'Environment': 'Home'
# }

# print("Mencari rekomendasi untuk user baru...")
# print(f"Profil: {new_user}")

# # 2. Cari ID User yang mau "dicontek"
# matched_id, score = find_similar_user(new_user)
# print(f"\n✅ Ditemukan Kembaran! User ID: {matched_id} (Skor Beda: {score})")
# print("Mengambil jadwal latihan dari User ID tersebut...\n")

# # 3. Tampilkan Jadwalnya
# rekomendasi_jadwal = get_workout_plan(matched_id)

# # Menampilkan per Hari biar rapi
# unique_days = rekomendasi_jadwal['Day'].unique()
# sorted_days = sorted(unique_days) # Mengurutkan Day 1, Day 2, dst

# for day in sorted_days:
#     print(f"📅 {day}")
#     day_plan = rekomendasi_jadwal[rekomendasi_jadwal['Day'] == day]
#     # Tampilkan kolom penting saja biar ga kepanjangan
#     display(day_plan[['Muscle Group', 'Exercise Name', 'Sets', 'Reps', 'Equipment']])
#     print("-" * 50)

In [72]:
import pandas as pd
import numpy as np
import pickle

In [73]:
import pickle
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# 1. LOAD THE OPTIMIZED MODEL
# ==========================================
path = '../../models/model_workout.pickle'

with open(path, 'rb') as f:
    model_data = pickle.load(f)
    
knn = model_data['knn_model']           # The Brain (Trained with Supernova Weights)
scaler = model_data['scaler']           # The Scaler (MinMax)
weights = model_data['weights']         # The Rules (Weights dictionary)
db_profiles = model_data['profiles_db'] # The User Database
db_schedule = model_data['schedule_db'] # The Workout Logs
encoders = model_data['encoders']       # The Encoders
feature_order = model_data['features']  # The Feature Map

print("✅ Workout Model Loaded & Ready.")

# ==========================================
# 2. PREDICTION FUNCTION (PRODUCTION READY)
# ==========================================

def predict_workout_plan(user_input):
    """
    Generates a personalized workout schedule using Weighted KNN.
    
    This function takes a new user's profile and constraints (Goal, Frequency, 
    Environment, Sports Capabilities) and finds the most similar user in the 
    database who STRICTLY matches those constraints.

    Args:
        user_input (dict): Dictionary containing user details:
            - 'Age' (int)
            - 'Gender' (str): 'Male' or 'Female'
            - 'Weight' (float): In kg
            - 'Goal' (str): 'Muscle Gain', 'Weight Loss', etc.
            - 'Frequency' (int): Days per week (e.g., 3, 4, 5)
            - 'Level' (str): 'Beginner', 'Intermediate', 'Advanced'
            - 'Environment' (str): 'Home' or 'Gym'
            - 'Badminton', 'Football', 'Basketball', 'Volleyball', 'Swim' (int): 
               1 if user can play/do this sport, 0 otherwise.

    Returns:
        DataFrame: The workout schedule of the matched user.
        (Also prints the Match Report to console for debugging).
    """
    
    # --- STEP 1: ENCODE USER INPUT ---
    # Convert text inputs (e.g., 'Male', 'Home') into numbers (0, 1, 2...)
    try:
        goal_enc = encoders['goal'].transform([user_input['Goal']])[0]
        level_enc = encoders['level'].transform([user_input['Level']])[0]
        gender_enc = encoders['gender'].transform([user_input['Gender']])[0]
        env_enc = encoders['environment'].transform([user_input['Environment']])[0]
    except ValueError as e:
        print(f"❌ Error: Invalid Category found. {e}")
        return None

    # --- STEP 2: PREPARE FEATURE VECTOR ---
    # Map inputs to the exact column order the model expects
    input_data = {
        'Goal_Encoded': goal_enc,
        'Workout_Frequency_x': user_input['Frequency'],
        'level_Encoded': level_enc,
        'Gender_Encoded': gender_enc,
        'Age_x': user_input['Age'],
        'Initial_Weight_kg_x': user_input['Weight'],
        'environment_Encoded': env_enc,
        
        # Cardio Constraints (Binary Flags 1/0)
        'Badminton': user_input['Badminton'],
        'Football': user_input['Football'],
        'Basketball': user_input['Basketball'],
        'Volleyball': user_input['Volleyball'],
        'Swim': user_input['Swim']
    }
    
    # Convert to DataFrame
    input_df = pd.DataFrame([input_data])[feature_order]
    
    # --- STEP 3: SCALE & WEIGHT ---
    # A. Scale features to 0-1 range
    input_scaled = scaler.transform(input_df)
    
    # B. Apply "Supernova Weights" (Multiply by 100 where needed)
    # This enforces the strict rules (Frequency, Environment, Sports)
    input_weighted = pd.DataFrame(input_scaled, columns=feature_order)
    for col, weight in weights.items():
        if col in input_weighted.columns:
            input_weighted[col] = input_weighted[col] * weight
            
    # --- STEP 4: FIND NEAREST NEIGHBOR ---
    # Find the single most similar user in the weighted space
    distances, indices = knn.kneighbors(input_weighted.values, n_neighbors=1)
    
    matched_index = indices[0][0]
    matched_user = db_profiles.iloc[matched_index]
    matched_user_id = matched_user['User_ID']
    
    # --- DEBUG REPORT ---
    print("\n" + "="*40)
    print(f"🔎 PLAN FOUND FOR: {user_input['Goal']} ({user_input['Frequency']} Days)")
    print("-" * 40)
    print(f"✅ MATCHED USER ID : {matched_user_id}")
    print(f"   • Goal Match    : {matched_user['Goal_x']}")
    print(f"   • Freq Match    : {matched_user['Workout_Frequency_x']} Days")
    print(f"   • Env Match     : {matched_user['Environment']}")
    print(f"   • Swim Ability  : {matched_user['Swim']} (User: {user_input['Swim']})")
    print("="*40)
    
    # --- STEP 5: RETRIEVE SCHEDULE ---
    schedule = db_schedule[db_schedule['User_ID'] == matched_user_id].copy()
    
    if 'Day' in schedule.columns:
        schedule = schedule.sort_values('Day')
        
    return schedule

# ==========================================
# 3. EXAMPLE USAGE
# ==========================================
if __name__ == "__main__":
    # Test User
    test_user = {
        'Age': 25, 'Gender': 'Male', 'Weight': 70,
        'Goal': 'Muscle Gain', 'Frequency': 3, 
        'Level': 'Beginner', 'Environment': 'Home',
        'Badminton': 0, 'Football': 1, 'Basketball': 0, 
        'Volleyball': 0, 'Swim': 1 
    }

    # Run Prediction
    plan = predict_workout_plan(test_user)
    
    # Display Result
    if plan is not None:
        cols = ['Day', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps']
        for day in plan['Day'].unique():
            print(f"\n📅 {day}")
            display(plan[plan['Day'] == day][cols].reset_index(drop=True))

✅ Workout Model Loaded & Ready.

🔎 PLAN FOUND FOR: Muscle Gain (3 Days)
----------------------------------------
✅ MATCHED USER ID : 289
   • Goal Match    : Muscle Gain
   • Freq Match    : 3 Days
   • Env Match     : Home
   • Swim Ability  : 1 (User: 1)

📅 Day 1 - Push & Burn


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 1 - Push & Burn,Chest,decline push up,3,8-12
1,Day 1 - Push & Burn,Cardio,Running in Place,1,24 Mins
2,Day 1 - Push & Burn,Cardio,Running in Place,1,24 Mins
3,Day 1 - Push & Burn,Triceps,impossible dips,3,8-12
4,Day 1 - Push & Burn,Shoulders,inchworm,3,8-12
...,...,...,...,...,...
60,Day 1 - Push & Burn,Cardio,Running in Place,1,24 Mins
61,Day 1 - Push & Burn,Triceps,impossible dips,3,8-12
62,Day 1 - Push & Burn,Shoulders,inchworm,3,8-12
63,Day 1 - Push & Burn,Chest,decline push up,3,8-12



📅 Day 2 - Pull & Core


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 2 - Pull & Core,Abs,side plank,3,8-12
1,Day 2 - Pull & Core,Back,superman,3,8-12
2,Day 2 - Pull & Core,Back,superman,3,8-12
3,Day 2 - Pull & Core,Abs,side plank,3,8-12
4,Day 2 - Pull & Core,Abs,side plank,3,8-12
5,Day 2 - Pull & Core,Back,superman,3,8-12
6,Day 2 - Pull & Core,Abs,side plank,3,8-12
7,Day 2 - Pull & Core,Abs,side plank,3,8-12
8,Day 2 - Pull & Core,Back,superman,3,8-12
9,Day 2 - Pull & Core,Abs,side plank,3,8-12



📅 Day 3 - Leg Power


,Day,Muscle Group,Exercise Name,Sets,Reps
0,Day 3 - Leg Power,Quads,bodyweight squat,3,8-12
1,Day 3 - Leg Power,Quads,bodyweight squat,3,8-12
2,Day 3 - Leg Power,Quads,reverse lunge,3,8-12
3,Day 3 - Leg Power,Hamstrings,single leg deadlift,3,8-12
4,Day 3 - Leg Power,Glutes,glute bridge,3,8-12
...,...,...,...,...,...
60,Day 3 - Leg Power,Quads,reverse lunge,3,8-12
61,Day 3 - Leg Power,Quads,bodyweight squat,3,8-12
62,Day 3 - Leg Power,Quads,bodyweight squat,3,8-12
63,Day 3 - Leg Power,Calves,standing calf raise,3,8-12
